# 01. Tree traversals

**Tier:** Deep. **Why you'd use this:** A binary tree should look the same when mirrored left-right. You compare the left child of one side with the right child of the other, recursively, until every pair matches or a mismatch appears.

**After this lesson you can:** recurse on mirrored child pairs; treat two missing nodes as a match and one missing as a miss.
About 25 minutes. **Uses:** [Classes](../../course/11-classes/01-classes-lesson.ipynb), [Recursion](../../course/09-recursion/01-base-case-lesson.ipynb)


## Quick review

1. What are the left and right attributes on a binary tree node?
2. When does a recursive walk on a tree stop?

<details><summary>Answers</summary>

Child node references, or `None`. It stops on a base case — usually a missing node. Skim [Recursion](../../course/09-recursion/01-base-case-lesson.ipynb) if the call stack picture is fuzzy.
</details>


## Recognize it

You must compare structure and values across a binary tree in a mirrored way. The prompt says "symmetric tree", "mirror of itself", or "left looks like right flipped".

Signals: recurse on `(left.left, right.right)` and `(left.right, right.left)`. A look-alike that is not this pattern: checking two trees are identical — that compares `(left.left, right.left)` without crossing.


## Why it works this way

A tree is a recursive structure: each node owns two smaller trees. Symmetry is also recursive. The root's left and right subtrees must be mirrors of each other. Two nodes `a` and `b` are mirrors when their values match, `a.left` mirrors `b.right`, and `a.right` mirrors `b.left`. Both missing is success; exactly one missing is failure.

An iterative level-order check can compare queues of mirrored children. The recursive form matches the definition directly. Time visits each node a constant number of times — proportional to the number of nodes. Extra memory is the call stack, which grows with the height (about `log n` for balanced trees, `n` for a stick).

Preorder, inorder, and postorder are different schedules for visiting a single tree: root-left-right, left-root-right, left-right-root. Symmetry is not a single-tree schedule; it is a paired walk. Same-tree comparison is the non-crossing cousin of this sibling.
Level-order traversal uses a queue of nodes and processes one depth at a time — the BFS schedule for trees. Invert-tree swaps left and right at every node; after an invert, symmetry checks become same-tree checks against the original shape in some puzzles. Validate-BST walks with a narrowing low/high bound rather than a mirror test.

Encoding a tree as a string with null markers lets you compare shapes by string equality. That is handy for serialization problems and is heavier than the direct recursive mirror walk when you only need a boolean. Prefer the boolean recursion until you must ship a durable encoding.


## When to use it

Use a mirrored recursive walk when the question is symmetry. Prefer identical-tree recursion when both trees should match without flipping. Prefer an explicit stack or queue when recursion depth might blow the limit on skewed inputs. Prefer storing one traversal string only when you need a fingerprint and accept the encoding cost.
 If only values matter and shape is fixed by the problem, comparing two inorder walks can suffice — but shape bugs hide there, so prefer structural recursion for symmetry.


## Brute force, and why it's slow

Build the left-to-right value list and the right-to-left value list with null markers, then compare the lists. Correct, but allocates two full snapshots.


In [1]:
class Node:
    def __init__(self, value, left=None, right=None):
        self.value = value
        self.left = left
        self.right = right


def record_view(node, left_first, out):
    if node is None:
        out.append(None)
        return
    out.append(node.value)
    if left_first:
        record_view(node.left, left_first, out)
        record_view(node.right, left_first, out)
    else:
        record_view(node.right, left_first, out)
        record_view(node.left, left_first, out)


def is_symmetric_slow(node):
    left_view: list = []
    right_view: list = []
    if node is None:
        return True, 0
    record_view(node.left, True, left_view)
    record_view(node.right, False, right_view)
    return left_view == right_view, len(left_view) + len(right_view)


tree = Node(1, Node(2, Node(3), Node(4)), Node(2, Node(4), Node(3)))
print(is_symmetric_slow(tree))
print(is_symmetric_slow(Node(1, Node(2), Node(3))))


(True, 14)
(False, 6)


## Key insight

Two nodes mirror each other when values match and their crossed children mirror each other. Fold that check from the root's left and right children.


## Worked example

Return whether a binary tree is symmetric around its center.


In [2]:
class Node:
    def __init__(self, value, left=None, right=None):
        self.value = value
        self.left = left
        self.right = right


def is_mirror(left, right) -> bool:
    # 1. Both missing matches. One missing does not.
    if left is None or right is None:
        return left is right
    # 2. Values match, and children cross-match.
    return (
        left.value == right.value
        and is_mirror(left.left, right.right)
        and is_mirror(left.right, right.left)
    )


def is_symmetric(node) -> bool:
    if node is None:
        return True
    return is_mirror(node.left, node.right)


print(is_symmetric(Node(1, Node(2, Node(3), Node(4)), Node(2, Node(4), Node(3)))))
print(is_symmetric(Node(1, Node(2), Node(3))))
print(is_symmetric(None))


True
False
True


## Trace

Tree `1` with left `2(3,4)` and right `2(4,3)`:

| pair | values | next pairs |
|---|---|---|
| (2, 2) | equal | (3, 3) and (4, 4) |
| (3, 3) | equal | (None, None) × 2 |
| (4, 4) | equal | (None, None) × 2 |

All match → **True**. Tree `1` with left `2` and right `3` fails at the first pair.


## Cost

Each node is visited a constant number of times: time grows with the node count. Extra memory is the recursion depth (tree height). The snapshot walk builds two lists of size proportional to the tree.

On a perfectly balanced tree the stack depth stays near `log n`, which is why recursion is comfortable here. On a linked-list-shaped tree the depth is `n` and you may prefer an explicit stack. The asymptotic time stays linear either way; only the constant factors and crash risk change.


## Variants

Not solved here: [Maximum Depth of Binary Tree](../../challenges/problems/27-maximum-depth-of-binary-tree.ipynb), [Invert Binary Tree](../../challenges/problems/28-invert-binary-tree.ipynb), [Binary Tree Level Order Traversal](../../challenges/problems/30-binary-tree-level-order-traversal.ipynb).


## Common mistake

This bug is silent. The recursion compares `left.left` with `right.left` (same side) instead of crossing. A mirrored tree is rejected. There is no traceback. The print is `False`, and the answer is `True`.


In [3]:
def is_mirror_wrong(left, right):
    if left is None or right is None:
        return left is right
    return (
        left.value == right.value
        and is_mirror_wrong(left.left, right.left)  # should cross
        and is_mirror_wrong(left.right, right.right)
    )


def is_symmetric_wrong(node):
    if node is None:
        return True
    return is_mirror_wrong(node.left, node.right)


print(is_symmetric_wrong(Node(1, Node(2, Node(3), Node(4)), Node(2, Node(4), Node(3)))))


False


The fix crosses the children: `left.left` with `right.right`, and `left.right` with `right.left`. Run `is_symmetric` on that mirrored tree and you get `True`.


In [4]:
print(is_symmetric(Node(1, Node(2, Node(3), Node(4)), Node(2, Node(4), Node(3)))))


True


## Python idioms

`left is right` covers both-`None` after an `or` guard: if either side is missing, they match only when they are the same object identity — both `None`. An explicit `if left is None and right is None` pair is clearer for some readers.


In [5]:
def is_mirror_explicit(left, right) -> bool:
    if left is None and right is None:
        return True
    if left is None or right is None:
        return False
    return (
        left.value == right.value
        and is_mirror_explicit(left.left, right.right)
        and is_mirror_explicit(left.right, right.left)
    )


print(is_mirror_explicit(Node(2, Node(3), Node(4)), Node(2, Node(4), Node(3))))


True


## Recap

- Symmetry is a mirrored pair walk, not a single preorder.
- Cross the children: left-left with right-right.
- Two missing nodes match; one missing node fails.

Next: [practice](01-same-tree-practice.ipynb).
